In [1]:
"""
Edge FC Matrix - Identity Lookup and Annotation
Helps identify which edges correspond to which positions in the matrix
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load your data
stats = pd.read_csv('../outputs/edgelist/edge_statistics.csv')
eFC = np.load('../outputs/edgelist/edge_fc_matrix.npy')

print("="*70)
print("EDGE IDENTITY LOOKUP TOOLS")
print("="*70)

# ============================================================================
# 1. INTERACTIVE LOOKUP FUNCTION
# ============================================================================

def lookup_edge(edge_index):
    """Look up identity of an edge by its index"""
    if edge_index < 0 or edge_index >= len(stats):
        print(f"❌ Edge index {edge_index} out of range (0-{len(stats)-1})")
        return
    
    edge = stats.iloc[edge_index]
    print(f"\n{'='*70}")
    print(f"EDGE {edge_index}")
    print(f"{'='*70}")
    print(f"Label: {edge['edge_label']}")
    print(f"Source → Target: {edge['source_node']} → {edge['target_node']}")
    
    if 'original_weight' in edge:
        print(f"Original weight: {edge['original_weight']:.2f}")
    if 'degree' in edge:
        print(f"Degree (# connections): {edge['degree']:.0f}")
    if 'betweenness' in edge:
        print(f"Betweenness centrality: {edge['betweenness']:.2f}")
    if 'community' in edge:
        print(f"Community: {edge['community']}")
    
    # Find what this edge connects to
    connections = np.where(eFC[edge_index, :] == 1)[0]
    print(f"\nConnects to {len(connections)} other edges")
    if len(connections) > 0 and len(connections) <= 10:
        print("Connected edges:")
        for conn_idx in connections[:10]:
            print(f"  • {stats.iloc[conn_idx]['edge_label'][:60]}")
    elif len(connections) > 10:
        print(f"Top 10 connected edges:")
        for conn_idx in connections[:10]:
            print(f"  • {stats.iloc[conn_idx]['edge_label'][:60]}")
        print(f"  ... and {len(connections)-10} more")


def check_connection(edge_i, edge_j):
    """Check if two edges are connected"""
    connected = eFC[edge_i, edge_j] == 1
    
    edge1 = stats.iloc[edge_i]
    edge2 = stats.iloc[edge_j]
    
    print(f"\n{'='*70}")
    print(f"CONNECTION CHECK: Edge {edge_i} ←→ Edge {edge_j}")
    print(f"{'='*70}")
    print(f"Edge {edge_i}: {edge1['edge_label'][:55]}")
    print(f"Edge {edge_j}: {edge2['edge_label'][:55]}")
    print(f"\nConnected: {'✓ YES' if connected else '✗ NO'}")
    
    if connected:
        # Find shared nodes
        source1, target1 = edge1['source_node'], edge1['target_node']
        source2, target2 = edge2['source_node'], edge2['target_node']
        
        shared = []
        if source1 == source2:
            shared.append(f"Same source: {source1}")
        if target1 == target2:
            shared.append(f"Same target: {target1}")
        if source1 == target2:
            shared.append(f"Sequential: {source1}")
        if target1 == source2:
            shared.append(f"Sequential: {target1}")
        
        if shared:
            print(f"Reason: {', '.join(shared)}")


# ============================================================================
# 2. CREATE ANNOTATED MATRIX REGIONS
# ============================================================================

def create_annotated_region(start_idx, end_idx, max_labels=20):
    """Create annotated matrix showing edge identities"""
    
    subset = eFC[start_idx:end_idx, start_idx:end_idx]
    n = subset.shape[0]
    
    fig, ax = plt.subplots(figsize=(16, 14))
    
    im = ax.imshow(subset, cmap='binary', aspect='auto', interpolation='nearest')
    
    # Add labels (but not too many or it's unreadable)
    label_step = max(1, n // max_labels)
    
    tick_positions = []
    tick_labels = []
    
    for i in range(0, n, label_step):
        actual_idx = start_idx + i
        if actual_idx < len(stats):
            tick_positions.append(i)
            # Get abbreviated label
            edge_info = stats.iloc[actual_idx]
            label = f"{actual_idx}: {edge_info['source_node'][:15]}→{edge_info['target_node'][:15]}"
            tick_labels.append(label)
    
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, rotation=90, fontsize=8, ha='right')
    ax.set_yticks(tick_positions)
    ax.set_yticklabels(tick_labels, fontsize=8)
    
    ax.set_xlabel('Edge (Source → Target)', fontsize=12)
    ax.set_ylabel('Edge (Source → Target)', fontsize=12)
    ax.set_title(f'Edge FC Matrix - Annotated Region\nEdges {start_idx} to {end_idx}',
                 fontsize=14, fontweight='bold', pad=20)
    
    plt.colorbar(im, ax=ax, label='Connected (1) / Not Connected (0)')
    
    plt.tight_layout()
    filename = f'eFC_annotated_edges_{start_idx}_to_{end_idx}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved: {filename}")
    return filename


# ============================================================================
# 3. FIND SPECIFIC EDGES
# ============================================================================

def find_edges_by_nodes(source_pattern=None, target_pattern=None):
    """Find edges involving specific nodes"""
    
    mask = pd.Series([True] * len(stats))
    
    if source_pattern:
        mask &= stats['source_node'].str.contains(source_pattern, case=False, na=False)
    
    if target_pattern:
        mask &= stats['target_node'].str.contains(target_pattern, case=False, na=False)
    
    results = stats[mask]
    
    print(f"\n{'='*70}")
    print(f"SEARCH RESULTS")
    print(f"{'='*70}")
    print(f"Query: source='{source_pattern}', target='{target_pattern}'")
    print(f"Found: {len(results)} edges")
    print(f"\n{'Index':<8} {'Source → Target':<60}")
    print("-"*70)
    
    for idx, row in results.head(20).iterrows():
        actual_idx = stats.index.get_loc(idx)
        print(f"{actual_idx:<8} {row['edge_label'][:60]}")
    
    if len(results) > 20:
        print(f"... and {len(results)-20} more")
    
    return results


# ============================================================================
# 4. VISUALIZE SPECIFIC EDGE AND ITS CONNECTIONS
# ============================================================================

def visualize_edge_connections(edge_idx, context=50):
    """Visualize a specific edge and what it connects to"""
    
    edge = stats.iloc[edge_idx]
    connections = np.where(eFC[edge_idx, :] == 1)[0]
    
    print(f"\nVisualizing Edge {edge_idx}: {edge['edge_label'][:60]}")
    print(f"Connects to {len(connections)} edges")
    
    # Create figure with multiple panels
    fig = plt.figure(figsize=(18, 6))
    
    # Panel 1: Row view (this edge to all others)
    ax1 = plt.subplot(1, 3, 1)
    row_view = eFC[edge_idx, :].reshape(1, -1)
    ax1.imshow(row_view, cmap='binary', aspect='auto')
    ax1.axvline(edge_idx, color='red', linewidth=2, label=f'Edge {edge_idx}')
    ax1.set_xlabel('Edge Index')
    ax1.set_title(f'Edge {edge_idx} connections\n(row view)', fontweight='bold')
    ax1.legend()
    
    # Panel 2: Column view (all others to this edge)
    ax2 = plt.subplot(1, 3, 2)
    col_view = eFC[:, edge_idx].reshape(-1, 1)
    ax2.imshow(col_view, cmap='binary', aspect='auto')
    ax2.axhline(edge_idx, color='red', linewidth=2, label=f'Edge {edge_idx}')
    ax2.set_ylabel('Edge Index')
    ax2.set_title(f'Edge {edge_idx} connections\n(column view)', fontweight='bold')
    ax2.legend()
    
    # Panel 3: Local context
    ax3 = plt.subplot(1, 3, 3)
    start = max(0, edge_idx - context)
    end = min(eFC.shape[0], edge_idx + context)
    local = eFC[start:end, start:end]
    
    im = ax3.imshow(local, cmap='binary', aspect='auto')
    
    # Mark the focal edge
    local_pos = edge_idx - start
    ax3.axhline(local_pos, color='red', linewidth=2, alpha=0.7)
    ax3.axvline(local_pos, color='red', linewidth=2, alpha=0.7)
    ax3.plot([local_pos], [local_pos], 'r*', markersize=20, label=f'Edge {edge_idx}')
    
    ax3.set_xlabel('Edge Index')
    ax3.set_ylabel('Edge Index')
    ax3.set_title(f'Local context\n(±{context} edges)', fontweight='bold')
    ax3.legend()
    
    plt.tight_layout()
    filename = f'edge_{edge_idx}_connections.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved: {filename}")
    return filename


# ============================================================================
# EXAMPLE USAGE
# ============================================================================

print("\n" + "="*70)
print("EXAMPLE USAGE")
print("="*70)

# Example 1: Look up a specific edge
print("\nExample 1: Look up edge 100")
lookup_edge(100)

# Example 2: Check if two edges are connected
print("\nExample 2: Check connection between edges 100 and 150")
check_connection(100, 150)

# Example 3: Find edges involving CA1
print("\nExample 3: Find edges from CA1 Pyramidal cells")
ca1_edges = find_edges_by_nodes(source_pattern='CA1 Pyramidal')

# Example 4: Create annotated region
print("\nExample 4: Create annotated matrix for edges 0-100")
create_annotated_region(0, 100, max_labels=15)

# Example 5: Visualize specific edge
print("\nExample 5: Visualize edge 200 and its connections")
visualize_edge_connections(200, context=50)

print("\n" + "="*70)
print("TOOLS READY TO USE!")
print("="*70)
print("\nYou can now call:")
print("  • lookup_edge(index) - Get info about an edge")
print("  • check_connection(i, j) - See if edges i and j connect")
print("  • find_edges_by_nodes('CA1', 'CA3') - Search for specific edges")
print("  • create_annotated_region(start, end) - Make labeled matrix")
print("  • visualize_edge_connections(index) - See all connections for one edge")

EDGE IDENTITY LOOKUP TOOLS

EXAMPLE USAGE

Example 1: Look up edge 100

EDGE 100
Label: DG HIPROM→CA3 Axo Axonic
Source → Target: DG HIPROM → CA3 Axo Axonic
Original weight: -4.73

Connects to 49 other edges
Top 10 connected edges:
  • DG Granule→DG HIPROM
  • DG Granule→CA3 Axo Axonic
  • DG Semilunar Granule→DG HIPROM
  • DG Semilunar Granule→CA3 Axo Axonic
  • DG Mossy→DG HIPROM
  • DG AIPRIM→DG HIPROM
  • DG HICAP→DG HIPROM
  • DG HIPROM→DG Granule
  • DG HIPROM→DG Semilunar Granule
  • DG HIPROM→DG Mossy
  ... and 39 more

Example 2: Check connection between edges 100 and 150

CONNECTION CHECK: Edge 100 ←→ Edge 150
Edge 100: DG HIPROM→CA3 Axo Axonic
Edge 150: DG Total Molecular Layer→DG MOPP

Connected: ✗ NO

Example 3: Find edges from CA1 Pyramidal cells

SEARCH RESULTS
Query: source='CA1 Pyramidal', target='None'
Found: 24 edges

Index    Source → Target                                             
----------------------------------------------------------------------
378      C